In [20]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("classification.pdf")
documents = loader.load()

print(len(documents))  # 看看有多少页
print(documents[0].page_content[:200])  # 看第一页内容

59
Introduction Logistic Regression k-Nearest Neighbor Decision Trees Naive Bayes Linear Discriminant Analysis Support Vector Machine Model Assessment References
Introduction to Data Science
Classiﬁcatio


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每块500字符
    chunk_overlap=100    # 重叠100字符（防止信息断裂）
)

split_docs = text_splitter.split_documents(documents)

print(len(split_docs))  # 会变很多块
print(split_docs[0].page_content)

101
Introduction Logistic Regression k-Nearest Neighbor Decision Trees Naive Bayes Linear Discriminant Analysis Support Vector Machine Model Assessment References
Introduction to Data Science
Classiﬁcation and nonlinear models
Zhen Zhang
Southern University of Science and Technology


In [22]:
import dashscope
from dashscope import TextEmbedding

# ============ DashScope API 配置 ============
# 注意：DashScope 的 API key 格式通常是 "sk-" 开头，不含 "-sp-"
# 请前往 https://dashscope.console.aliyun.com/apiKey 获取正确的 key
API_KEY = "sk-b0f15faccf3f46cdb915d8d472e405a5"  # ← 如果报错，请替换为正确的 key
MODEL_NAME = "text-embedding-v2"

print(f"正在测试 API key: {API_KEY[:15]}...")

# 测试 API - 使用正确的调用方式
try:
    dashscope.api_key = API_KEY
    response = TextEmbedding.call(
        model=MODEL_NAME,
        input="测试文本"
    )
    
    print("完整响应：", response)
    
    if response.status_code == 200:
        print("✅ API 密钥有效！")
        print("Embedding 维度：", len(response.output['embeddings'][0]['embedding']))
        
        def get_embedding(text):
            """获取文本的 embedding 向量"""
            if isinstance(text, list):
                texts = text
            else:
                texts = [text]
            
            resp = TextEmbedding.call(model=MODEL_NAME, input=texts)
            if resp.status_code == 200:
                embeddings = [item['embedding'] for item in resp.output['embeddings']]
                return embeddings[0] if len(embeddings) == 1 else embeddings
            else:
                print(f"Embedding 失败：{resp.code} - {resp.message}")
                return None
    else:
        print("❌ API key 无效！")
        print(f"错误代码：{response.code}")
        print(f"错误信息：{response.message}")
        print("\n请前往 https://dashscope.console.aliyun.com/apiKey 获取正确的 API Key")
        
except Exception as e:
    print("发生异常：", e)

正在测试 API key: sk-b0f15faccf3f...
完整响应： {"status_code": 200, "request_id": "736c7170-c836-9c4e-b5d4-eb467e593416", "code": "", "message": "", "output": {"embeddings": [{"embedding": [-0.01957997002816164, 0.07229063607010255, 0.032889526252077514, -0.0074800308902811495, 0.030402463084528118, -0.020650160845713198, -0.017319003512207637, -0.018524852320716437, -0.029995489111656396, -0.0039491548478663195, 0.031653531223356, -0.028277154559531358, 0.0330101111329284, -0.00524544231701328, 0.01826860944890832, 0.02043913730422416, 0.010332616977909779, 0.0005779595656407412, -0.030749144616974398, 0.038707746753132474, -0.01148570990104632, -0.01456816091779694, 0.008456014769667959, 0.027945546137191437, -0.012073561195194359, -0.01864543720156732, -0.01279707048029964, -0.0034988456834388146, -0.025413263639322958, 0.0343666910425008, 0.014198869720191118, 0.02616691914464096, 0.00852384376514658, 0.024358145931877757, 0.00676028988270246, 0.011078735928174599, 0.012412706172587459, 0.

In [46]:
from langchain_community.vectorstores import FAISS
import numpy as np

# 为所有 split_docs 生成 embeddings
print("正在生成 embeddings...")
texts = [doc.page_content for doc in split_docs]

# 批量获取 embeddings
all_embeddings = []
for i, text in enumerate(texts):
    emb = get_embedding(text)
    if emb:
        all_embeddings.append(emb)
    if (i + 1) % 10 == 0:
        print(f"已处理 {i + 1}/{len(texts)} 个文档...")

print(f"✅ 生成了 {len(all_embeddings)} 个 embeddings")
embedding_dim = len(all_embeddings[0])

# 创建一个简单的 embedding 函数类（用于后续查询）
class DashScopeEmbeddings:
    def __init__(self, dim):
        self.embedding_dimension = dim
    
    def embed_documents(self, texts):
        result = []
        for text in texts:
            emb = get_embedding(text)
            if emb:
                result.append(emb)
        return result
    
    def embed_query(self, text):
        return get_embedding(text)
    
    def __call__(self, text):
        """使对象可调用，兼容 FAISS 内部调用"""
        return self.embed_query(text)

embeddings_model = DashScopeEmbeddings(embedding_dim)

# 从预计算的 embeddings 创建 FAISS
# text_embeddings 参数需要是 [(text1, embedding1), (text2, embedding2), ...] 格式
text_embedding_pairs = list(zip(
    [doc.page_content for doc in split_docs[:len(all_embeddings)]],
    all_embeddings
))

vectorstore = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=embeddings_model,
    metadatas=[doc.metadata for doc in split_docs[:len(all_embeddings)]]
)

print("✅ FAISS 向量存储创建成功！")

# 保存 FAISS 索引到本地，供 rag.py 使用
vectorstore.save_local("faiss_index")
print("✅ FAISS 索引已保存到 faiss_index 文件夹")

正在生成 embeddings...
已处理 10/101 个文档...
已处理 20/101 个文档...
已处理 30/101 个文档...
已处理 40/101 个文档...
已处理 50/101 个文档...
已处理 60/101 个文档...
已处理 70/101 个文档...
已处理 80/101 个文档...
已处理 90/101 个文档...
已处理 100/101 个文档...


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


✅ 生成了 101 个 embeddings
✅ FAISS 向量存储创建成功！
✅ FAISS 索引已保存到 faiss_index 文件夹


In [40]:
# 初始化 Anthropic client - 在调用 ask_rag 之前必须先运行这个
from anthropic import Anthropic

# 替换为你的 Anthropic API key
client = Anthropic(api_key="YOUR_ANTHROPIC_API_KEY")

print("✅ Anthropic client 初始化完成")

✅ Anthropic client 初始化完成


In [ ]:
# 使用 DashScope 调用通义千问模型（阿里云百炼）
from http import HTTPStatus
import dashscope

# 你的阿里云 DashScope API key
DASHSCOPE_API_KEY = "sk-b0f15faccf3f46cdb915d8d472e405a5"

# 定义 RAG 查询函数（带引用来源）
def ask_rag(question, k=5):
    """基于向量搜索结果回答问题，并标注引用来源"""
    docs = vectorstore.similarity_search(question, k=k)
    
    # 实际检索到的文档数量
    actual_count = len(docs)
    
    # 构建带引用编号的上下文
    context_parts = []
    for i, doc in enumerate(docs):
        ref_num = i + 1
        content = doc.page_content.strip()
        # 截取每段内容的前 300 字符，避免太长
        if len(content) > 300:
            content = content[:300] + "..."
        context_parts.append(f"[{ref_num}] {content}")
    
    context = "\n\n".join(context_parts)
    
    # 构建来源列表
    sources = []
    for i, doc in enumerate(docs):
        ref_num = i + 1
        # 从 page_content 开头提取来源标识（前 50 字符）
        source_hint = doc.page_content[:80].strip().replace("\n", " ")
        if len(source_hint) > 80:
            source_hint = source_hint[:80] + "..."
        sources.append(f"[{ref_num}] {source_hint}")
    
    sources_text = "\n".join(sources)
    
    prompt = f"""你是一个机器学习助教，请根据以下带编号的资料回答问题。
要求：
1. 用简洁清晰的中文回答
2. 尽量用通俗语言解释概念
3. 如果资料不足，请回答“我不知道”

资料（共 {actual_count} 条）：
{context}

问题：{question}

请用简洁的中文回答，并在引用处标注编号，格式如"【来源 1】"。
注意：只有 {actual_count} 条资料，所以只能引用 [1]{"到【来源" + str(actual_count) + "】" if actual_count > 1 else "】"}，不要编造不存在的来源。

回答："""

    # 使用 DashScope 调用通义千问模型
    response = dashscope.Generation.call(
        model="qwen-plus",
        prompt=prompt,
        api_key=DASHSCOPE_API_KEY
    )
    
    if response.status_code == HTTPStatus.OK:
        answer = response.output.text
        # 拼接来源列表
        full_response = f"{answer}\n\n来源：\n{sources_text}"
        return full_response
    else:
        print(f"❌ 请求失败:")
        print(f"   状态码：{response.status_code}")
        print(f"   错误代码：{response.code}")
        print(f"   错误信息：{response.message}")
        return None

# 测试
result = ask_rag("what is kNN regression")
if result:
    print(result)

kNN回归是用k近邻法做回归预测：对新样本，找训练集中距离最近的k个邻居，然后用这k个邻居的目标变量的平均值作为预测结果【来源 1】。

来源：
[1] Introduction Logistic Regression k-Nearest Neighbor Decision Trees Naive Bayes L
[2] Introduction Logistic Regression k-Nearest Neighbor Decision Trees Naive Bayes L
[3] Introduction Logistic Regression k-Nearest Neighbor Decision Trees Naive Bayes L


In [45]:
import streamlit as st

st.title("📚 机器学习 RAG 问答系统")

# 输入框
question = st.text_input("请输入你的问题：")

# 你的RAG函数（先写个假的，等会再接真的）
def ask_rag(q):
    return f"你问的是：{q}"

# 点击或输入后执行
if question:
    answer = ask_rag(question)
    st.write("### 🤖 回答：")
    st.write(answer)